In [2]:
!pip install -q transformers peft datasets accelerate huggingface_hub scikit-learn
!pip install -q -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 54.3 MB/s eta 0:00:00


In [3]:
from huggingface_hub import notebook_login
notebook_login()

In [4]:
import json
import urllib.request

GITHUB_RAW_BASE = "https://raw.githubusercontent.com/Adigit2211/RBI_Multilingual_RAG/main/finetune"

def load_jsonl_from_url(url):
    data = []
    with urllib.request.urlopen(url) as response:
        for line in response:
            data.append(json.loads(line))
    return data

train_data = load_jsonl_from_url(f"{GITHUB_RAW_BASE}/train.jsonl")
val_data = load_jsonl_from_url(f"{GITHUB_RAW_BASE}/val.jsonl")

print(f"Train: {len(train_data)} examples")
print(f"Val: {len(val_data)} examples")
print(train_data[0])

Train: 109 examples
Val: 27 examples
{'text': 'RBI/202 6-27/185 DOR.STR.REC. 158/21 -04-048/2026 -27                   July 15, 2026 Reserve Bank of India ( Urban Cooperative Banks – Credit Facilities) Second Amendment Directions, 2026 Please refer to Reserve Bank of India ( Urban Cooperative Banks  – Credit Facilities) Directions, 2025 (hereinafter referred to as ‘the Directions’).', 'label': 'DOR'}


In [5]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Build label <-> id mappings from the training set
labels = sorted(set(d["label"] for d in train_data))
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}
print("Labels:", label2id)

def tokenize_examples(data):
    texts = [d["text"] for d in data]
    label_ids = [label2id[d["label"]] for d in data]
    encodings = tokenizer(texts, truncation=True, padding=True, max_length=512, return_tensors="pt")
    encodings["labels"] = label_ids
    return encodings

train_encodings = tokenize_examples(train_data)
val_encodings = tokenize_examples(val_data)

print("Train tokenized shape:", train_encodings["input_ids"].shape)
print("Val tokenized shape:", val_encodings["input_ids"].shape)

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

Labels: {'DOR': 0, 'DOS': 1, 'FIDD': 2, 'OTHER': 3}
Train tokenized shape: torch.Size([109, 381])
Val tokenized shape: torch.Size([27, 302])


In [6]:
import torch

class CircularsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __len__(self):
        return len(self.encodings["labels"])

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items() if k != "labels"}
        item["labels"] = torch.tensor(self.encodings["labels"][idx])
        return item

train_dataset = CircularsDataset(train_encodings)
val_dataset = CircularsDataset(val_encodings)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Val dataset size: {len(val_dataset)}")


Train dataset size: 109
Val dataset size: 27


In [8]:
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)

# LoRA config: we only train small low-rank adapter matrices injected into
# the attention layers (q_lin, v_lin in DistilBERT), not the full model.
# This is why LoRA fine-tuning is feasible on a free Colab T4 and produces
# a tiny (~few MB) artifact instead of a full ~500MB fine-tuned checkpoint.
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,                  # rank of the low-rank matrices -- small since our dataset is tiny
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],  # DistilBERT's attention projection layers
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()


model.safetensors: reconstructing file:   0%|          |  0.00B /  542MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 741,124 || all params: 136,068,872 || trainable%: 0.5447


In [9]:
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def compute_metrics(eval_pred):
    logits, label_ids = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(label_ids, predictions),
        "f1_macro": f1_score(label_ids, predictions, average="macro"),
    }

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-4,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=5,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

In [10]:
trainer.train()

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.018624,0.716161,0.888889,0.460606
2,0.521916,0.467841,0.888889,0.460606
3,0.266151,0.471467,0.851852,0.440092
4,0.234439,0.457342,0.851852,0.440092
5,0.450471,0.446330,0.888889,0.460606
6,0.273986,0.424139,0.888889,0.625000
7,0.126522,0.386586,0.925926,0.721429
8,0.164019,0.389351,0.925926,0.721429
9,0.211305,0.391587,0.925926,0.721429
10,0.107103,0.393933,0.925926,0.721429


TrainOutput(global_step=140, training_loss=0.3444681853055954, metrics={'train_runtime': 34.9159, 'train_samples_per_second': 31.218, 'train_steps_per_second': 4.01, 'total_flos': 109296586994400.0, 'train_loss': 0.3444681853055954, 'epoch': 10.0})

In [13]:
HF_USERNAME = "Aditideo"  # your HF username from the login
REPO_NAME = "rbi-circular-department-classifier-lora"

model.push_to_hub(f"{HF_USERNAME}/{REPO_NAME}")
tokenizer.push_to_hub(f"{HF_USERNAME}/{REPO_NAME}")

print(f"Pushed to: https://huggingface.co/{HF_USERNAME}/{REPO_NAME}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  19%|#8        |  557kB / 2.97MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Pushed to: https://huggingface.co/Aditideo/rbi-circular-department-classifier-lora


In [14]:
from sklearn.metrics import classification_report

predictions = trainer.predict(val_dataset)
pred_labels = np.argmax(predictions.predictions, axis=-1)
true_labels = predictions.label_ids

print(classification_report(
    true_labels, pred_labels,
    target_names=[id2label[i] for i in range(len(labels))],
    zero_division=0
))

              precision    recall  f1-score   support

         DOR       0.88      1.00      0.93        14
         DOS       1.00      0.91      0.95        11
        FIDD       1.00      1.00      1.00         1
       OTHER       0.00      0.00      0.00         1

    accuracy                           0.93        27
   macro avg       0.72      0.73      0.72        27
weighted avg       0.90      0.93      0.91        27



In [15]:
HF_USERNAME = "Aditideo"
REPO_NAME = "rbi-circular-department-classifier-lora"

model.push_to_hub(f"{HF_USERNAME}/{REPO_NAME}")
tokenizer.push_to_hub(f"{HF_USERNAME}/{REPO_NAME}")

print(f"Pushed to: https://huggingface.co/{HF_USERNAME}/{REPO_NAME}")


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors: 100%|##########| 2.97MB / 2.97MB            

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


Pushed to: https://huggingface.co/Aditideo/rbi-circular-department-classifier-lora
